# Limpeza e Preparação de Dados - Passos Mágicos Datathon

## Objetivo
Padronizar, limpar e unificar os dados de três anos (2022-2024) para uma análise abrangente.

## Processo
1. Padronizar nomes de colunas entre os anos
2. Remover colunas com 100% de ausência (artefatos de metadata)
3. Consolidar colunas dos indicadores principais
4. Criar um dataset unificado multi-ano
5. Validar e documentar as transformações

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuração inicial
project_root = Path.cwd().parent
DATA_FILE = project_root / "data" / "raw" / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
DATA_DIR = project_root / "data"
DATA_DIR.mkdir(exist_ok=True)

print(f"Diretório de trabalho: {project_root}")
print(f"Diretório de dados: {DATA_DIR}")

Diretório de trabalho: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5
Diretório de dados: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data


## 1. Carregar Dados Brutos

In [2]:
# Carregar todos os sheets
data_dict = {}
excel_file = pd.ExcelFile(DATA_FILE)

for sheet in excel_file.sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet)
    data_dict[sheet] = df
    print(f"{sheet}: {df.shape[0]} linhas x {df.shape[1]} colunas")

PEDE2022: 860 linhas x 42 colunas
PEDE2023: 1014 linhas x 48 colunas
PEDE2024: 1156 linhas x 50 colunas


## 2. Definir Mapeamento de Colunas

Mapear variações de nomes de colunas para nomes canônicos.

In [3]:
# Colunas canônicas - indicadores principais presentes em todos os anos
CANONICAL_COLUMNS = {
    # Identificadores
    'RA': 'student_id',
    'Fase': 'phase',
    'Turma': 'class',
    'Nome': 'name',
    
    # Dados demográficos
    'Ano nasc': 'birth_year',
    'Idade 22': 'age_2022',
    'Idade': 'age',
    'Gênero': 'gender',
    'Ano ingresso': 'admission_year',
    'Instituição de ensino': 'school_institution',
    'Escola': 'school_institution',
    
    # Indicadores centrais (foco principal da análise)
    'IAN': 'ian',  # Adequação acadêmica
    'IDA': 'ida',  # Desempenho acadêmico
    'IEG': 'ieg',  # Engajamento
    'IAA': 'iaa',  # Autoavaliação
    'IPS': 'ips',  # Aspectos psicossociais
    'IPP': 'ipp',  # Aspectos psicopedagógicos
    'IPV': 'ipv',  # Ponto de virada
    'INDE 22': 'inde_2022',
    'INDE 23': 'inde_2023',
    'INDE 2023': 'inde_2023',
    'INDE 2024': 'inde_2024',
    'INDE 24': 'inde_2024',
    
    # Disciplinas acadêmicas
    'Matem': 'math',
    'Mat': 'math',
    'Portug': 'portuguese',
    'Por': 'portuguese',
    'Inglês': 'english',
    'Ing': 'english',
    
    # Indicador de defasagem
    'Defas': 'deficiency',
    'IAN': 'ian',
    
    # Ponto de virada
    'Atingiu PV': 'achieved_turning_point',
    'Indicado': 'indicated_for_intervention',
}

# Colunas a excluir (100% faltantes ou artefatos de metadados)
COLUMNS_TO_EXCLUDE = {
    'Destaque IPV.1', 'Destaque IPV', 'Destaque IEG', 'Destaque IDA',  # Duplicadas/artefatos
    'Rec Av1', 'Rec Av2', 'Rec Av3', 'Rec Av4',  # Colunas de recomendação 100% faltantes
    'Rec Psicologia',  # 100% faltante
    'Avaliador5', 'Avaliador6',  # 87-99% faltantes, inexistentes em anos anteriores
    'Cg', 'Cf', 'Ct',  # Sistema interno de pontuação sem uso nesta análise
    'Nº Av',  # Número de avaliadores - metadado
    'Avaliador1', 'Avaliador2', 'Avaliador3', 'Avaliador4',  # Nomes de avaliadores não necessários
    'Pedra 20', 'Pedra 21', 'Pedra 22', 'Pedra 23', 'Pedra 2024',  # Marcadores de etapa fora do foco analítico
    'Pedra 23',
}

print(f"Colunas canônicas definidas: {len(CANONICAL_COLUMNS)}")
print(f"Colunas a excluir: {len(COLUMNS_TO_EXCLUDE)}")

Colunas canônicas definidas: 32
Colunas a excluir: 24


## 3. Limpar Dados por Ano

In [4]:
def clean_year_data(df, year_label):
    """
    Limpa os dados de um único ano:
    1. Renomeia colunas
    2. Remove colunas excluídas
    3. Adiciona identificador de ano
    4. Mantém apenas colunas analisáveis
    """
    df_clean = df.copy()
    
    # Remover colunas duplicadas primeiro (manter primeira ocorrência)
    df_clean = df_clean.loc[:, ~df_clean.columns.duplicated(keep='first')]
    
    # Filtrar colunas que devem ser renomeadas
    cols_to_rename = {col: CANONICAL_COLUMNS[col] for col in df_clean.columns if col in CANONICAL_COLUMNS}
    df_clean.rename(columns=cols_to_rename, inplace=True)

    # Remover duplicatas geradas pelo rename (ex.: INDE 2023 e INDE 23)
    # Mantemos a primeira ocorrência, que no PEDE2023 é a coluna oficial INDE 2023.
    df_clean = df_clean.loc[:, ~df_clean.columns.duplicated(keep='first')]
    
    # Remover colunas excluídas
    cols_to_drop = [col for col in df_clean.columns if col in COLUMNS_TO_EXCLUDE]
    df_clean.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    
    # Remover colunas com 100% de ausência (exceto indicadores esperados)
    missing_pct = (df_clean.isnull().sum() / len(df_clean) * 100)
    cols_100_missing = missing_pct[missing_pct == 100.0].index.tolist()
    # Não remover indicadores centrais/INDE mesmo se faltantes (podem existir em outros anos)
    protected_columns = ['ian', 'ida', 'ieg', 'iaa', 'ips', 'ipp', 'ipv', 'inde_2022', 'inde_2023', 'inde_2024']
    cols_100_missing = [col for col in cols_100_missing if col not in protected_columns]
    df_clean.drop(columns=cols_100_missing, inplace=True, errors='ignore')
    
    # Remover colunas duplicadas que possam surgir após etapas de limpeza
    df_clean = df_clean.loc[:, ~df_clean.columns.duplicated(keep='first')]
    
    # Adicionar identificador de ano
    df_clean['year'] = year_label
    
    return df_clean

# Limpar cada ano
data_cleaned = {}
for year_label, df in data_dict.items():
    df_clean = clean_year_data(df, year_label)
    data_cleaned[year_label] = df_clean
    print(f"\n{year_label}:")
    print(f"  Original: {df.shape[1]} colunas")
    print(f"  Limpo: {df_clean.shape[1]} colunas")
    print(f"  Linhas restantes: {df_clean.shape[0]}")
    # Verificar se ainda há colunas duplicadas
    if df_clean.columns.duplicated().any():
        print(f"  ALERTA: Colunas duplicadas detectadas!")


PEDE2022:
  Original: 42 colunas
  Limpo: 24 colunas
  Linhas restantes: 860

PEDE2023:
  Original: 48 colunas
  Limpo: 25 colunas
  Linhas restantes: 1014

PEDE2024:
  Original: 50 colunas
  Limpo: 27 colunas
  Linhas restantes: 1156


## 4. Padronizar e Mesclar Anos

In [5]:
# Obter todas as colunas únicas nos datasets limpos
all_columns = set()
for df in data_cleaned.values():
    all_columns.update(df.columns)

print(f"Todas as colunas únicas entre os anos: {len(all_columns)}")
print(f"Colunas: {sorted(all_columns)}")

Todas as colunas únicas entre os anos: 35
Colunas: ['Ativo/ Inativo', 'Ativo/ Inativo.1', 'Data de Nasc', 'Defasagem', 'Fase Ideal', 'Fase ideal', 'Nome Anonimizado', 'Pedra 2023', 'achieved_turning_point', 'admission_year', 'age', 'age_2022', 'birth_year', 'class', 'deficiency', 'english', 'gender', 'iaa', 'ian', 'ida', 'ieg', 'inde_2022', 'inde_2023', 'inde_2024', 'indicated_for_intervention', 'ipp', 'ips', 'ipv', 'math', 'name', 'phase', 'portuguese', 'school_institution', 'student_id', 'year']


In [6]:
# Remover colunas duplicadas e criar conjunto padronizado
for year_label in data_cleaned.keys():
    df = data_cleaned[year_label]
    # Manter apenas colunas sem sufixos .1, .2 (duplicatas)
    df_clean = df.loc[:, ~df.columns.str.contains(r'\.\d+$', regex=True)]
    data_cleaned[year_label] = df_clean

# Obter colunas únicas limpas
all_columns_clean = set()
for df in data_cleaned.values():
    all_columns_clean.update(df.columns)

print(f"Colunas únicas limpas: {len(all_columns_clean)}")
print(f"Colunas: {sorted(all_columns_clean)}")

# Garantir que todos os dataframes tenham as mesmas colunas
for year_label in data_cleaned.keys():
    for col in all_columns_clean:
        if col not in data_cleaned[year_label].columns:
            data_cleaned[year_label][col] = np.nan

# Mesclar todos os anos
dfs_to_concat = []
for year_label in ['PEDE2022', 'PEDE2023', 'PEDE2024']:
    df_temp = data_cleaned[year_label].copy()
    df_temp = df_temp.reset_index(drop=True)
    dfs_to_concat.append(df_temp)

df_unified = pd.concat(dfs_to_concat, ignore_index=True, sort=False)

print(f"\nDataset unificado: {df_unified.shape[0]} linhas x {df_unified.shape[1]} colunas")
print(f"Anos representados: {df_unified['year'].unique()}")
print(f"\nLinhas por ano:")
print(df_unified['year'].value_counts().sort_index())

Colunas únicas limpas: 34
Colunas: ['Ativo/ Inativo', 'Data de Nasc', 'Defasagem', 'Fase Ideal', 'Fase ideal', 'Nome Anonimizado', 'Pedra 2023', 'achieved_turning_point', 'admission_year', 'age', 'age_2022', 'birth_year', 'class', 'deficiency', 'english', 'gender', 'iaa', 'ian', 'ida', 'ieg', 'inde_2022', 'inde_2023', 'inde_2024', 'indicated_for_intervention', 'ipp', 'ips', 'ipv', 'math', 'name', 'phase', 'portuguese', 'school_institution', 'student_id', 'year']

Dataset unificado: 3030 linhas x 34 colunas
Anos representados: ['PEDE2022' 'PEDE2023' 'PEDE2024']

Linhas por ano:
year
PEDE2022     860
PEDE2023    1014
PEDE2024    1156
Name: count, dtype: int64


## 5. Validar Indicadores Principais

In [7]:
# Verificar disponibilidade dos indicadores principais
core_indicators = ['ian', 'ida', 'ieg', 'iaa', 'ips', 'ipp', 'ipv']

print("Disponibilidade dos Indicadores Principais:")
print("="*80)

for indicator in core_indicators:
    if indicator in df_unified.columns:
        missing_pct = (df_unified[indicator].isnull().sum() / len(df_unified) * 100)
        stats = df_unified[indicator].describe()
        print(f"\n{indicator.upper()}:")
        print(f"  Faltantes: {df_unified[indicator].isnull().sum()} ({missing_pct:.1f}%)")
        print(f"  Intervalo: {stats['min']:.1f} - {stats['max']:.1f}")
        print(f"  Média: {stats['mean']:.2f}, Desvio Padrão: {stats['std']:.2f}")
    else:
        print(f"\n{indicator.upper()}: NAO ENCONTRADO")

Disponibilidade dos Indicadores Principais:

IAN:
  Faltantes: 0 (0.0%)
  Intervalo: 2.5 - 10.0
  Média: 7.18, Desvio Padrão: 2.54

IDA:
  Faltantes: 178 (5.9%)
  Intervalo: 0.0 - 10.0
  Média: 6.38, Desvio Padrão: 1.96

IEG:
  Faltantes: 76 (2.5%)
  Intervalo: 0.0 - 10.0
  Média: 7.95, Desvio Padrão: 2.15

IAA:
  Faltantes: 165 (5.4%)
  Intervalo: 0.0 - 10.0
  Média: 7.92, Desvio Padrão: 2.63

IPS:
  Faltantes: 171 (5.6%)
  Intervalo: 2.5 - 10.0
  Média: 6.29, Desvio Padrão: 1.79

IPP:
  Faltantes: 1038 (34.3%)
  Intervalo: 2.5 - 10.0
  Média: 7.56, Desvio Padrão: 0.94

IPV:
  Faltantes: 178 (5.9%)
  Intervalo: 2.5 - 10.0
  Média: 7.55, Desvio Padrão: 1.08


## 6. Tratar Valores Faltantes nos Indicadores Principais

Para os indicadores principais, apenas documentar os valores faltantes e manter os registros por enquanto.

In [8]:
# Linhas com indicadores principais faltantes por ano
print("Linhas com Indicadores Principais Faltantes por Ano:")
print("="*80)

for year in ['PEDE2022', 'PEDE2023', 'PEDE2024']:
    df_year = df_unified[df_unified['year'] == year]
    print(f"\n{year}:")
    
    for indicator in core_indicators:
        if indicator in df_year.columns:
            missing = df_year[indicator].isnull().sum()
            missing_pct = (missing / len(df_year) * 100)
            print(f"  {indicator}: {missing} ({missing_pct:.1f}%)")

Linhas com Indicadores Principais Faltantes por Ano:

PEDE2022:
  ian: 0 (0.0%)
  ida: 0 (0.0%)
  ieg: 0 (0.0%)
  iaa: 0 (0.0%)
  ips: 0 (0.0%)
  ipp: 860 (100.0%)
  ipv: 0 (0.0%)

PEDE2023:
  ian: 0 (0.0%)
  ida: 77 (7.6%)
  ieg: 76 (7.5%)
  iaa: 63 (6.2%)
  ips: 69 (6.8%)
  ipp: 76 (7.5%)
  ipv: 76 (7.5%)

PEDE2024:
  ian: 0 (0.0%)
  ida: 101 (8.7%)
  ieg: 0 (0.0%)
  iaa: 102 (8.8%)
  ips: 102 (8.8%)
  ipp: 102 (8.8%)
  ipv: 102 (8.8%)


## 7. Salvar Datasets Limpos

In [9]:
# Dataset intermediario mantido em memoria (df_unified)
# O arquivo oficial salvo ao final e a versao padronizada em data/dados_unificados.csv

# Salvar anos individuais (para referência)
for year_label, df_year_clean in data_cleaned.items():
    output_path = DATA_DIR / f"dados_{year_label.lower()}.csv"
    df_year_clean.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Salvo: {output_path}")

# Construir versao padronizada para consumo analitico/modelagem

def normalize_year(series):
    s = series.astype(str).str.strip().str.upper()
    return s.where(s.str.startswith('PEDE'), np.nan)

def normalize_gender(series):
    m = {
        'MENINA': 'Feminino',
        'FEMININO': 'Feminino',
        'MENINO': 'Masculino',
        'MASCULINO': 'Masculino'
    }
    return series.astype(str).str.strip().str.upper().map(m)

def normalize_school(series):
    s = series.astype(str).str.strip()
    def mapper(v):
        low = v.lower()
        if 'public' in low or 'pública' in low:
            return 'Escola Pública'
        if 'rede decis' in low:
            return 'Rede Decisão'
        if 'privad' in low or 'bols' in low or 'empresa parceira' in low:
            return 'Escola Privada'
        return 'Outros'
    return s.map(mapper)

def normalize_yes_no(series):
    m = {
        'SIM': 'Sim',
        'NAO': 'Não',
        'NAO': 'Não',
        'YES': 'Sim',
        'NO': 'Não',
        '1': 'Sim',
        '0': 'Não',
        'TRUE': 'Sim',
        'FALSE': 'Não'
    }
    return series.astype(str).str.strip().str.upper().map(m)


df_canonical = df_unified.copy()
df_canonical['year'] = normalize_year(df_canonical['year'])
year_num = pd.to_numeric(df_canonical['year'].astype(str).str[-4:], errors='coerce')

age = pd.to_numeric(df_canonical.get('age'), errors='coerce')
age_2022 = pd.to_numeric(df_canonical.get('age_2022'), errors='coerce')

df_canonical['age_canonical'] = age
derive_age = df_canonical['age_canonical'].isna() & age_2022.notna() & year_num.notna()
df_canonical.loc[derive_age, 'age_canonical'] = age_2022[derive_age] + (year_num[derive_age] - 2022)

df_canonical['age_2022_canonical'] = age_2022
derive_age_2022 = df_canonical['age_2022_canonical'].isna() & age.notna() & year_num.notna()
df_canonical.loc[derive_age_2022, 'age_2022_canonical'] = age[derive_age_2022] - (year_num[derive_age_2022] - 2022)

df_canonical['ideal_phase'] = df_canonical.get('Fase Ideal').combine_first(df_canonical.get('Fase ideal'))
df_canonical['gender'] = normalize_gender(df_canonical['gender'])
df_canonical['school_institution'] = normalize_school(df_canonical['school_institution'])
df_canonical['achieved_turning_point'] = normalize_yes_no(df_canonical['achieved_turning_point'])
df_canonical['indicated_for_intervention'] = normalize_yes_no(df_canonical['indicated_for_intervention'])

deficiency_raw = pd.to_numeric(df_canonical.get('deficiency'), errors='coerce')
df_canonical['deficiency_bin'] = np.where(deficiency_raw > 0, 1.0, np.where(deficiency_raw.notna(), 0.0, np.nan))

df_canonical['age'] = df_canonical['age_canonical']
df_canonical['age_2022'] = df_canonical['age_2022_canonical']
df_canonical['deficiency'] = df_canonical['deficiency_bin']
df_canonical['phase'] = df_canonical['phase'].astype(str).str.strip()
df_canonical.loc[df_canonical['phase'].isin(['', 'nan', 'None']), 'phase'] = np.nan

numeric_cols = [
    'admission_year', 'age', 'age_2022', 'ian', 'ida', 'ieg', 'iaa', 'ips', 'ipv', 'ipp',
    'math', 'portuguese', 'english', 'inde_2022', 'inde_2023', 'inde_2024', 'deficiency'
]
for col in numeric_cols:
    if col in df_canonical.columns:
        df_canonical[col] = pd.to_numeric(df_canonical[col], errors='coerce')

# Remover colunas auxiliares temporárias antes de salvar a versão oficial
helper_cols = ['age_canonical', 'age_2022_canonical', 'deficiency_bin']
df_canonical = df_canonical.drop(columns=[c for c in helper_cols if c in df_canonical.columns])

output_path_unified = DATA_DIR / 'dados_unificados.csv'
df_canonical.to_csv(output_path_unified, index=False, encoding='utf-8')
print(f"Dataset unificado padronizado salvo: {output_path_unified}")

print(f"\nLimpeza de dados concluída!")
print(f"\nDataset pronto para análise:")
print(f"  Total de registros: {len(df_canonical)}")
print(f"  Total de colunas: {df_canonical.shape[1]}")
print(f"  Indicadores principais presentes: {len([c for c in core_indicators if c in df_canonical.columns])}")
print(f"  Missing age: {df_canonical['age'].isna().mean():.1%}")
print(f"  Missing age_2022: {df_canonical['age_2022'].isna().mean():.1%}")

Salvo: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data\dados_pede2022.csv
Salvo: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data\dados_pede2023.csv
Salvo: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data\dados_pede2024.csv
Dataset unificado padronizado salvo: c:\Users\cso2569\Python\Pos-Tech-Data-Analytics\Modulo 5\fiap-datathon-fase5\data\dados_unificados.csv

Limpeza de dados concluída!

Dataset pronto para análise:
  Total de registros: 3030
  Total de colunas: 35
  Indicadores principais presentes: 7
  Missing age: 13.2%
  Missing age_2022: 13.2%


## 8. Resumo

Preparação de dados concluída.

Artefatos oficiais produzidos neste notebook:
- `data/dados_unificados.csv`: base oficial padronizada para analise e modelagem.
- `data/dados_pede2022.csv`, `data/dados_pede2023.csv`, `data/dados_pede2024.csv`: bases anuais limpas.

A versão padronizada resolve redundâncias semânticas da consolidação, como:
- `age` e `age_2022`.
- `Fase ideal` e `Fase Ideal`.
- categorias inconsistentes em `gender`, `school_institution` e flags binárias.